In [ ]:
# 1_filtering.ipynb
# Cell tag parameters (for batch processing — papermill reads these and loops through the participants)
# PAPERMILL_RUN is toggled "True" if batch processing is being used
p_id = 'participant_1'
PAPERMILL_RUN = False

In [ ]:
# Imports
import mne
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Allow imports to work regardless of the notebook's location by locating the project root
# EEG_DIR is defined in config.py
def find_project_root(current_path, marker='config.py'):
    path = Path(current_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path
root_dir = find_project_root(Path.cwd())
sys.path.append(str(root_dir))

from config import EEG_DIR

mne.set_log_level('error') # reduce extraneous MNE output

# Plotting Toggle: it is deactivated if running each notebook manually, but if batch processing
# is being used, it is activated so windows don't pop-up
if PAPERMILL_RUN:
    # Batch Mode: Suppress pop-ups and draw plots inline
    %matplotlib inline
    mne.viz.set_browser_backend('matplotlib')
    show_plots = False
else:
    # Manual Mode: Allow interactive pop-ups for data exploration
    %matplotlib auto
    show_plots = True


In [ ]:
# Path Management
subj_dir = EEG_DIR / p_id # each participant's folder
raw_file = subj_dir / f'{p_id}.vhdr'

In [ ]:
# Read raw EEG data
raw = mne.io.read_raw_brainvision(raw_file, preload=True)

# Set electrode locations
montage = 'easycap-M1' # Electrode position file
raw.set_montage(montage)

In [ ]:
# Filter settings
low_cut = 0.1
hi_cut = 30

raw_filt = raw.copy().filter(low_cut, hi_cut)

In [ ]:
# PSD of the filtered data
raw_filt.compute_psd().plot(show=show_plots);

In [ ]:
# Generate a plot where the max frequency is 5 (to better observe the effects of the high pass filter)
raw_filt.compute_psd(fmax=5).plot(show=show_plots);

In [ ]:
# Plot the unfiltered waves
raw.plot(start=15, duration=5, show=show_plots); # times are in seconds

In [ ]:
# Plot the filtered waves
raw_filt.plot(start=15, duration=5, show=show_plots); # times are in seconds

In [ ]:
# Save the filtered data to a .fif file
raw_filt.save(subj_dir / f'{p_id}-filt-raw.fif', overwrite=True)

# Save the unfiltered data to a .fif file
raw.save(subj_dir / f'{p_id}-raw.fif', overwrite=True);

In [ ]:
print(f"✅ Filtering complete for {p_id}")